# Alternative $\psi$ initial guess

In previous examples using the static inverse solver, the solver has been initialised with an initial guess for $\psi$. This initial guess is used in the first iteration of the solver which attempts to find a solution of $\psi$ that satisfies the Grad-Shafranov equation.

---

We start off by defining some helper functions to construct `Equilibrium` using different initial flux guesses. This is very similar to code found in previous notebooks showing off the inverse solver.

Note that the function `create_equilibrium_obj` accepts a `psi` argument which is directly passed onto the `Equilibrium` constructor.

In [ ]:
from freegsnke import build_machine
from freegsnke import equilibrium_update


def create_equilibrium_obj(psi = None):
    tokamak = build_machine.tokamak(
        active_coils_path="../machine_configs/MAST-U/MAST-U_like_active_coils.pickle",
        passive_coils_path="../machine_configs/MAST-U/MAST-U_like_passive_coils.pickle",
        limiter_path="../machine_configs/MAST-U/MAST-U_like_limiter.pickle",
        wall_path="../machine_configs/MAST-U/MAST-U_like_wall.pickle",
    )

    eq = equilibrium_update.Equilibrium(
        tokamak=tokamak,      # provide tokamak object
        Rmin=0.1, Rmax=2.0,   # radial range
        Zmin=-2.2, Zmax=2.2,  # vertical range
        nx=65,                # number of grid points in the radial direction (needs to be of the form (2**n + 1) with n being an integer)
        ny=129,               # number of grid points in the vertical direction (needs to be of the form (2**n + 1) with n being an integer)
        psi=psi
    )

    eq.tokamak.set_coil_current('Solenoid', 5000)
    eq.tokamak['Solenoid'].control = False 

    return eq

In [ ]:
import numpy as np

def plot_eq_psi(eq, *, levels=15):
    ax = eq.tokamak.plot(show=False)
    ax.set_aspect("equal")

    psi = eq.psi()
    levels = np.linspace(np.min(psi), np.max(psi), 8)
    ax.contour(eq.R, eq.Z, psi, levels=levels)

    return ax

## Gaussian guess

The default initial guess uses a Gaussian-shaped flux function

$$
\psi(R, Z) = \text{exp}(C - (|R - R_c|^p + |Z - Z_c|^p))
$$
where $(R_c, Z_c)$ are the cylindrical coordinates of the flux centre, $C$ is a shift parameter in the exponent, and $p$ is the power of the distance term.

In [ ]:
import matplotlib.pyplot as plt

eq_default_gaussian = create_equilibrium_obj()

plot_eq_psi(eq_default_gaussian)
plt.show()

The constants that parameterise the Gaussian approximation can be modified by specifying `gpars=(Rc, Zc, C, p)`.

In [ ]:
from freegsnke.equilibrium import PsiGuessGaussian

eq_gaussian = create_equilibrium_obj(psi=PsiGuessGaussian(gpars=(0.5, 0.5, 0.01, 1.2)))
plot_eq_psi(eq_gaussian)
plt.show()

The `PsiGuessGaussian` also supports an `adaptive_centre` parameter which automatically finds $R_c$ and $Z_c$.

## Parabolic guess

This model assumes the plasma boundary can be described by an ellipse and contains a parabolic distribution of current within the ellipse. 

An elliptic distribution $E(R, Z)$ is found such that $E(h, k) = 0$ at the centre of the distribution $(h, k)$ and $E(R, Z) = 1$ for all $R$ and $Z$ on the plasma boundary. 

The distribution of plasma current is described by 

$$
J_{\text{tor}}(R, Z) = \hat{J}(1-E(R,Z)^i)^j
$$
where $\hat{J}$ is found to match the prescribed $I_p$.

Applying a linear Grad-Shafranov step to this current distribution provides the initial flux guess. 

**Reference**

Wai, J. T., et al. "Feedforward equilibrium trajectory optimization with GSPulse." Nuclear Fusion 66.1 (2026): 016047.

There are two practical ways to use the parabolic guess model:

First, we can provide the location of the midplane sand X-point. The elliptic function is constructed to pass through the midplanes and X-point. This model assumes that $h$ is the mean of the radial coordinates of the inner midplane, X-point, and outer midplane.


In [ ]:
from freegsnke.equilibrium import PsiGuessParabolicJtor

Rx = 0.6
Zx=1.1
Ri=0.34
Ro=1.4
Zm=0.0
Ip=6e5


eq_parabolic_midplane_xpoint = create_equilibrium_obj(psi=PsiGuessParabolicJtor.from_xpoint_and_midplane(
    Rx=Rx, Zx=Zx, Ri=Ri, Ro=Ro, Zm=Zm, Ip=Ip
))
plot_eq_psi(eq_parabolic_midplane_xpoint)
plt.show()

Second, we can provide a series of point on the plasma boundary (separatrix). The parameters of the elliptic distribution are found to best match these boundary points. Remember, $E(R, Z) = 1$ on the boundary, however the provided separatrix points can likely not be perfectly described with an ellipse, hence a best fit is found with OLS.

In [ ]:
eq_parabolic_separatrix = create_equilibrium_obj(psi=PsiGuessParabolicJtor.from_separatrix_points(
    Ip,
    *(
        (Rx, Zx), (Rx, -Zx), (Ri, Zm), (Ro, Zm)
    )
))
plot_eq_psi(eq_parabolic_separatrix)
plt.show()

Hopefully, the more accurate $\psi$ guess provided by the parabolic model provides a _better_ starting point for the solver, enabling it to solve the equilibrium in less iterations. 

NOTE: this guess can also be used with the inverse solver to improve convergence.

In [ ]:
from freegsnke.jtor_update import ConstrainPaxisIp
from freegsnke import GSstaticsolver
import pickle

def static_solve_eq(eq):

    profiles = ConstrainPaxisIp(
        eq=eq,        # equilibrium object
        paxis=8e3,    # profile object
        Ip=Ip,       # plasma current
        fvac=0.5,     # fvac = rB_{tor}
        alpha_m=1.8,  # profile function parameter
        alpha_n=1.2   # profile function parameter
    )

    GSStaticSolver = GSstaticsolver.NKGSsolver(
        eq,
        gs_operator_order=4,  # use 2 for lower direct-solver setup cost
    )

    with open('data/simple_diverted_currents_PaxisIp.pk', 'rb') as f:
        currents_dict = pickle.load(f)
        
    # assign currents to the eq object
    for key in currents_dict:
        eq.tokamak.set_coil_current(coil_label=key, current_value=currents_dict[key])
        
    GSStaticSolver.solve(eq=eq, 
        profiles=profiles, 
        constrain=None, 
        target_relative_tolerance=1e-9,
        verbose=True, # print output
    )

    eq.plot()
    plt.show()


For example, the forward solve using the default Gaussian distribution...

In [ ]:
static_solve_eq(eq_default_gaussian)

... takes more iterations than using the parabolic current distribution model despite producing similar results.

In [ ]:
static_solve_eq(eq_parabolic_separatrix)